In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
pip install torch transformers torchaudio soundfile pandas numpy


Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import soundfile as sf
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2Model,
    AutoTokenizer,
    AutoModel,
)

In [5]:
for root, dirs, files in os.walk("/kaggle/input"):
    print(root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/sahajwagle
/kaggle/input/datasets/sahajwagle/daic-woz-dataset
/kaggle/input/datasets/sahajwagle/daic-woz-dataset/transcript
/kaggle/input/datasets/sahajwagle/daic-woz-dataset/splits
/kaggle/input/datasets/sahajwagle/daic-woz-dataset/audio
/kaggle/input/datasets/sahajwagle/daic-woz-processed


In [6]:
SPLITS_DIR      = "/kaggle/input/datasets/sahajwagle/daic-woz-dataset/splits"
PROCESSED_AUDIO = "/kaggle/input/datasets/sahajwagle/daic-woz-dataset/audio"
PROCESSED_TEXT  = "/kaggle/input/datasets/sahajwagle/daic-woz-dataset/transcript"

In [7]:
print(os.listdir(SPLITS_DIR))

['dev.csv', 'class_weights.csv', 'train.csv', 'test.csv']


In [8]:
OUTPUT_AUDIO_EMB = "/kaggle/working/embeddings/audio"
OUTPUT_TEXT_EMB  = "/kaggle/working/embeddings/text"

In [25]:
AUDIO_MODEL_NAME = "facebook/wav2vec2-base-960h"
TEXT_MODEL_NAME  = "mental/mental-bert-base-uncased"

In [10]:
MAX_TEXT_TOKENS = 512

In [11]:
MAX_AUDIO_SECONDS = 30

In [12]:
TARGET_SR = 16000 

In [13]:
os.makedirs(OUTPUT_AUDIO_EMB, exist_ok=True)
os.makedirs(OUTPUT_TEXT_EMB,  exist_ok=True)

In [14]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"Audio  : {PROCESSED_AUDIO}")
print(f"Text   : {PROCESSED_TEXT}")
print(f"Splits : {SPLITS_DIR}")

Device : cuda
Audio  : /kaggle/input/datasets/sahajwagle/daic-woz-dataset/audio
Text   : /kaggle/input/datasets/sahajwagle/daic-woz-dataset/transcript
Splits : /kaggle/input/datasets/sahajwagle/daic-woz-dataset/splits


In [15]:
print("Splits folder contents:")
if os.path.exists(SPLITS_DIR):
    for f in sorted(os.listdir(SPLITS_DIR)):
        size = os.path.getsize(os.path.join(SPLITS_DIR, f))
        print(f"  {f}  ({size:,} bytes)")
else:
    print(f"  ERROR: {SPLITS_DIR} not found")
 
# Check audio folder
print(f"\nAudio files found : {len(os.listdir(PROCESSED_AUDIO)) if os.path.exists(PROCESSED_AUDIO) else 'PATH NOT FOUND'}")
print(f"Transcript files  : {len(os.listdir(PROCESSED_TEXT)) if os.path.exists(PROCESSED_TEXT) else 'PATH NOT FOUND'}")

Splits folder contents:
  class_weights.csv  (20 bytes)
  dev.csv  (5,105 bytes)
  test.csv  (6,206 bytes)
  train.csv  (15,629 bytes)

Audio files found : 94
Transcript files  : 94


In [16]:
def load_all_participant_ids():
    """
    Read train/dev/test manifest CSVs and return sorted list of
    participant ids that have both a .wav and a .txt file on disk.
    """
    all_ids      = set()
    missing_files = []
 
    for split_name in ["train.csv", "dev.csv", "test.csv"]:
        path = os.path.join(SPLITS_DIR, split_name)
        if not os.path.exists(path):
            print(f"  WARNING: {path} not found, skipping")
            continue
 
        df = pd.read_csv(path)
        for _, row in df.iterrows():
            pid        = int(row["participant_id"])
            audio_path = os.path.join(PROCESSED_AUDIO, f"{pid}.wav")
            text_path  = os.path.join(PROCESSED_TEXT,  f"{pid}.txt")
 
            if os.path.exists(audio_path) and os.path.exists(text_path):
                all_ids.add(pid)
            else:
                missing_files.append(pid)
 
    if missing_files:
        unique_missing = sorted(set(missing_files))
        print(f"  NOTE: {len(unique_missing)} participant(s) skipped "
              f"(no processed files): {unique_missing[:10]}"
              + (" ..." if len(unique_missing) > 10 else ""))
 
    return sorted(all_ids)
 
 
def extract_audio_embedding(audio_path, processor, model):
    """
    Load .wav, run through wav2vec2 in chunks, return mean-pooled
    (768,) numpy array.
    """
    audio, sr = sf.read(audio_path)
    audio = audio.astype(np.float32)
 
    if audio.ndim == 2:           # collapse stereo to mono
        audio = audio.mean(axis=1)
 
    if sr != TARGET_SR:
        import torchaudio
        audio_tensor = torch.tensor(audio).unsqueeze(0)
        resampler    = torchaudio.transforms.Resample(orig_freq=sr, new_freq=TARGET_SR)
        audio        = resampler(audio_tensor).squeeze(0).numpy()
 
    chunk_size   = MAX_AUDIO_SECONDS * TARGET_SR
    chunk_embeds = []
 
    for start in range(0, len(audio), chunk_size):
        chunk = audio[start : start + chunk_size]
        if len(chunk) < 400:
            continue
 
        inputs = processor(
            chunk,
            sampling_rate=TARGET_SR,
            return_tensors="pt",
            padding=True,
        )
        input_values = inputs.input_values.to(DEVICE)
 
        with torch.no_grad():
            outputs = model(input_values)
 
        embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
        chunk_embeds.append(embedding.cpu().numpy())
 
    if not chunk_embeds:
        return None
 
    return np.mean(chunk_embeds, axis=0)
 
 
def extract_text_embedding(text_path, tokenizer, model):
    """
    Read .txt transcript, run through Mental-BERT in overlapping
    chunks, return mean-pooled (768,) numpy array.
    """
    with open(text_path, "r", encoding="utf-8") as f:
        text = f.read().strip()
 
    if not text:
        return None
 
    tokens      = tokenizer(text, add_special_tokens=False, return_tensors="pt")
    input_ids   = tokens["input_ids"][0]
    total_tokens = len(input_ids)
 
    chunk_len    = MAX_TEXT_TOKENS - 2
    stride       = chunk_len // 2
    chunk_embeds = []
 
    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id
 
    for start in range(0, total_tokens, stride):
        end       = min(start + chunk_len, total_tokens)
        chunk_ids = input_ids[start:end]
 
        chunk_ids = torch.cat([
            torch.tensor([cls_id]),
            chunk_ids,
            torch.tensor([sep_id]),
        ]).unsqueeze(0).to(DEVICE)
 
        attention_mask = torch.ones_like(chunk_ids)
 
        with torch.no_grad():
            outputs = model(input_ids=chunk_ids, attention_mask=attention_mask)
 
        cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze(0)
        chunk_embeds.append(cls_embedding.cpu().numpy())
 
        if end == total_tokens:
            break
 
    if not chunk_embeds:
        return None
 
    return np.mean(chunk_embeds, axis=0)
 
 
print("Helper functions defined.")

Helper functions defined.


In [17]:
print("Collecting participant IDs from manifests ...")
participant_ids = load_all_participant_ids()
print(f"\nTotal participants with processed files: {len(participant_ids)}")
print(f"ID range: {participant_ids[0]} to {participant_ids[-1]}")

  NOTE: 95 participant(s) skipped (no processed files): [300, 301, 302, 303, 304, 305, 306, 307, 308, 309] ...

Total participants with processed files: 94
ID range: 396 to 492


In [18]:
print(f"Loading audio model: {AUDIO_MODEL_NAME} ...")
audio_processor = Wav2Vec2Processor.from_pretrained(AUDIO_MODEL_NAME)
audio_model     = Wav2Vec2Model.from_pretrained(AUDIO_MODEL_NAME)
audio_model.eval()
audio_model.to(DEVICE)
print("Audio model loaded.\n")
 
done = skipped = failed = 0
 
for i, pid in enumerate(participant_ids, 1):
    out_path = os.path.join(OUTPUT_AUDIO_EMB, f"{pid}.npy")
 
    if os.path.exists(out_path):          # resume support
        skipped += 1
        continue
 
    audio_path = os.path.join(PROCESSED_AUDIO, f"{pid}.wav")
 
    try:
        embedding = extract_audio_embedding(audio_path, audio_processor, audio_model)
 
        if embedding is None:
            print(f"  [{i}/{len(participant_ids)}] {pid}  SKIPPED (no valid audio chunks)")
            failed += 1
            continue
 
        np.save(out_path, embedding)
        done += 1
        print(f"  [{i}/{len(participant_ids)}] {pid}  shape={embedding.shape}")
 
    except Exception as e:
        print(f"  [{i}/{len(participant_ids)}] {pid}  ERROR: {e}")
        failed += 1
 
print(f"\nAudio extraction complete.")
print(f"  Done: {done}  |  Skipped (already existed): {skipped}  |  Failed: {failed}")
 
# Free GPU memory before loading text model
del audio_model
del audio_processor
torch.cuda.empty_cache()
print("GPU memory freed.")

Loading audio model: facebook/wav2vec2-base-960h ...


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Audio model loaded.

  [1/94] 396  shape=(768,)
  [2/94] 397  shape=(768,)
  [3/94] 399  shape=(768,)
  [4/94] 400  shape=(768,)
  [5/94] 401  shape=(768,)
  [6/94] 402  shape=(768,)
  [7/94] 403  shape=(768,)
  [8/94] 404  shape=(768,)
  [9/94] 405  shape=(768,)
  [10/94] 406  shape=(768,)
  [11/94] 407  shape=(768,)
  [12/94] 408  shape=(768,)
  [13/94] 409  shape=(768,)
  [14/94] 410  shape=(768,)
  [15/94] 411  shape=(768,)
  [16/94] 412  shape=(768,)
  [17/94] 413  shape=(768,)
  [18/94] 414  shape=(768,)
  [19/94] 415  shape=(768,)
  [20/94] 416  shape=(768,)
  [21/94] 417  shape=(768,)
  [22/94] 418  shape=(768,)
  [23/94] 419  shape=(768,)
  [24/94] 420  shape=(768,)
  [25/94] 421  shape=(768,)
  [26/94] 422  shape=(768,)
  [27/94] 423  shape=(768,)
  [28/94] 424  shape=(768,)
  [29/94] 425  shape=(768,)
  [30/94] 426  shape=(768,)
  [31/94] 427  shape=(768,)
  [32/94] 428  shape=(768,)
  [33/94] 429  shape=(768,)
  [34/94] 430  shape=(768,)
  [35/94] 431  shape=(768,)
  [36/94

In [32]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_1 = user_secrets.get_secret("HF_TOKEN2")


In [33]:
print(f"Loading text model: {TEXT_MODEL_NAME} ...")
text_tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_model     = AutoModel.from_pretrained(TEXT_MODEL_NAME)
text_model.eval()
text_model.to(DEVICE)
print("Text model loaded.\n")
 
done = skipped = failed = 0
 
for i, pid in enumerate(participant_ids, 1):
    out_path = os.path.join(OUTPUT_TEXT_EMB, f"{pid}.npy")
 
    if os.path.exists(out_path):          # resume support
        skipped += 1
        continue
 
    text_path = os.path.join(PROCESSED_TEXT, f"{pid}.txt")
 
    try:
        embedding = extract_text_embedding(text_path, text_tokenizer, text_model)
 
        if embedding is None:
            print(f"  [{i}/{len(participant_ids)}] {pid}  SKIPPED (empty transcript)")
            failed += 1
            continue
 
        np.save(out_path, embedding)
        done += 1
        print(f"  [{i}/{len(participant_ids)}] {pid}  shape={embedding.shape}")
 
    except Exception as e:
        print(f"  [{i}/{len(participant_ids)}] {pid}  ERROR: {e}")
        failed += 1
 
print(f"\nText extraction complete.")
print(f"  Done: {done}  |  Skipped (already existed): {skipped}  |  Failed: {failed}")
 
del text_model
del text_tokenizer
torch.cuda.empty_cache()
print("GPU memory freed.")

Loading text model: mental/mental-bert-base-uncased ...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: mental/mental-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your d

Text model loaded.


Text extraction complete.
  Done: 0  |  Skipped (already existed): 94  |  Failed: 0
GPU memory freed.


In [34]:
print("Verifying embeddings ...\n")
issues = []
 
for pid in participant_ids:
    for label, folder in [("audio", OUTPUT_AUDIO_EMB), ("text", OUTPUT_TEXT_EMB)]:
        path = os.path.join(folder, f"{pid}.npy")
 
        if not os.path.exists(path):
            issues.append(f"  {pid}  MISSING {label} embedding")
            continue
 
        emb = np.load(path)
        if emb.shape != (768,):
            issues.append(f"  {pid}  WRONG {label} shape: {emb.shape} (expected (768,))")
 
if issues:
    print(f"{len(issues)} issue(s) found:")
    for issue in issues:
        print(issue)
else:
    print(f"All {len(participant_ids)} participants have valid audio + text embeddings (768,)")
 
print(f"\nEmbeddings saved to:")
print(f"  {OUTPUT_AUDIO_EMB}")
print(f"  {OUTPUT_TEXT_EMB}")
print(f"\nNext step: Save this notebook as a Kaggle dataset output,")
print(f"then use the embeddings for model training.")

Verifying embeddings ...

All 94 participants have valid audio + text embeddings (768,)

Embeddings saved to:
  /kaggle/working/embeddings/audio
  /kaggle/working/embeddings/text

Next step: Save this notebook as a Kaggle dataset output,
then use the embeddings for model training.
